In [33]:
# ═══════════════════════════════════════════════════════════════════════
# CYBERPATH v2 — AI-POWERED RECOMMENDATION SYSTEM

# This notebook builds an independent, supervised machine learning
# recommendation system trained on synthetic student-machine relevance data.
#
# It is SEPARATE from the v1 system (recommender.ipynb), which remains
# unchanged and continues to use dataset_final.csv.
#
# v2 uses the corrected dataset (dataset_v2_ai_clean.csv) and trains two
# supervised ranking models for comparison: Random Forest and XGBoost.

# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set up paths — relative to the notebook location
BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'data', 'dataset_v2_ai_clean.csv')
MODELS_DIR_V2 = os.path.join(BASE_DIR, 'saved_models', 'ai_recommender')

# Create the v2 models folder if it doesn't exist (won't touch v1's folder)
os.makedirs(MODELS_DIR_V2, exist_ok=True)
os.makedirs(os.path.join(MODELS_DIR_V2, 'plots'), exist_ok=True)

print(f"BASE_DIR:       {BASE_DIR}")
print(f"DATA_PATH:      {DATA_PATH}")
print(f"MODELS_DIR_V2:  {MODELS_DIR_V2}")
print(f"\nDataset file exists: {os.path.exists(DATA_PATH)}")
print(f"v2 folder created:   {os.path.exists(MODELS_DIR_V2)}")

BASE_DIR:       c:\curtin\project new\Vuln Recommendation system
DATA_PATH:      c:\curtin\project new\Vuln Recommendation system\data\dataset_v2_ai_clean.csv
MODELS_DIR_V2:  c:\curtin\project new\Vuln Recommendation system\saved_models\ai_recommender

Dataset file exists: True
v2 folder created:   True


In [34]:
# load dataset and check basics

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print("columns:", list(df.columns))

# any missing values?
print("\nmissing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# check distributions of main fields
print("\ndifficulty:")
print(df['Difficulty'].value_counts())

print("\nattack category:")
print(df['Attack_Category'].value_counts())

print("\nos:")
print(df['OS'].value_counts())

print("\nestimated time:")
print(df['Estimated_Time'].value_counts())

# how many unique learning objectives
all_objs = []
for s in df['Learning_Objectives'].dropna():
    all_objs.extend([x.strip() for x in str(s).split(';') if x.strip()])
print("\nunique learning objectives:", len(set(all_objs)))

# quick look at first 3 rows
df[['Machine_Name', 'Difficulty', 'Attack_Category', 'OS', 'Estimated_Time']].head(3)

shape: (305, 18)
columns: ['Machine_ID', 'Machine_Name', 'Platform', 'Platform_ID', 'OS', 'OS_Detail', 'Difficulty', 'Difficulty_Numeric', 'Attack_Category', 'Attack_Category_Detail', 'Vulnerability_Type', 'Kill_Chain_Stages', 'Skills_Required', 'Learning_Objectives', 'Estimated_Time', 'Estimated_Time_Hours', 'Attack_Path_Length', 'Entry_Point']

missing:
OS_Detail    67
dtype: int64

difficulty:
Difficulty
Medium    159
Hard      106
Easy       40
Name: count, dtype: int64

attack category:
Attack_Category
Web Exploitation              190
Network Exploitation          107
Binary Exploitation             4
Mixed (Web + Network)           3
Cryptographic Exploitation      1
Name: count, dtype: int64

os:
OS
Linux      303
FreeBSD      1
Windows      1
Name: count, dtype: int64

estimated time:
Estimated_Time
2-3 hours    140
1-2 hours    103
3-4 hours     43
4+ hours      11
<1 hour        8
Name: count, dtype: int64

unique learning objectives: 99


,Machine_Name,Difficulty,Attack_Category,OS,Estimated_Time
0,LAMPSecurity CTF 4,Easy,Web Exploitation,Linux,1-2 hours
1,LAMPSecurity CTF 5,Easy,Web Exploitation,Linux,2-3 hours
2,LAMPSecurity CTF 7,Easy,Web Exploitation,Linux,2-3 hours


In [35]:
sample = str(df['Kill_Chain_Stages'].iloc[0])
print("sample:", sample)
print("contains arrow:", '→' in sample)

sample: Recon → Exploitation → Credential Access → Lateral Movement → Privilege Escalation
contains arrow: True


In [36]:
# encode machines into numeric features for the ml model

from sklearn.preprocessing import MultiLabelBinarizer

# difficulty as number
diff_map = {'Easy': 1, 'Medium': 2, 'Hard': 3}
df['diff_num'] = df['Difficulty'].map(diff_map)

# count kill chain stages per machine
def count_chain_stages(s):
    if pd.isna(s): return 0
    return len([p for p in str(s).split('→') if p.strip()])

df['kill_chain_count'] = df['Kill_Chain_Stages'].apply(count_chain_stages)
print("kill chain count - min/max/mean:",
      df['kill_chain_count'].min(),
      df['kill_chain_count'].max(),
      round(df['kill_chain_count'].mean(), 2))

# attack category - one hot
attack_cat = pd.get_dummies(df['Attack_Category'], prefix='cat').astype(int)

# os - one hot
os_cat = pd.get_dummies(df['OS'], prefix='os').astype(int)

# learning objectives - multi hot
def split_objs(s):
    if pd.isna(s): return []
    return [x.strip() for x in str(s).split(';') if x.strip()]

obj_lists = df['Learning_Objectives'].apply(split_objs)
mlb = MultiLabelBinarizer()
obj_matrix = mlb.fit_transform(obj_lists)
obj_df = pd.DataFrame(obj_matrix, columns=['obj_' + c for c in mlb.classes_])

# final machine feature matrix
machine_features = pd.concat([
    df[['diff_num', 'Estimated_Time_Hours', 'Attack_Path_Length', 'kill_chain_count']].reset_index(drop=True),
    attack_cat.reset_index(drop=True),
    os_cat.reset_index(drop=True),
    obj_df.reset_index(drop=True)
], axis=1)

print("machine feature matrix shape:", machine_features.shape)
print("total objective columns:", len([c for c in machine_features.columns if c.startswith('obj_')]))

# keep clean machine info for displaying to user later
machine_info = df[['Machine_ID', 'Machine_Name', 'Platform', 'OS', 'Difficulty',
                   'Attack_Category', 'Vulnerability_Type', 'Entry_Point',
                   'Attack_Path_Length', 'Estimated_Time', 'Estimated_Time_Hours',
                   'Skills_Required', 'Learning_Objectives', 'Kill_Chain_Stages']].copy()

# save
joblib.dump(machine_features, os.path.join(MODELS_DIR_V2, 'machine_features.pkl'))
joblib.dump(machine_info, os.path.join(MODELS_DIR_V2, 'machine_info_v2.pkl'))
joblib.dump(mlb, os.path.join(MODELS_DIR_V2, 'obj_encoder.pkl'))

print("saved.")

kill chain count - min/max/mean: 3 7 4.99
machine feature matrix shape: (305, 111)
total objective columns: 99
saved.


In [37]:
# quick checks on the dataset before we move on

print("any duplicate machine names?", df['Machine_Name'].duplicated().sum())

# numeric ranges
print("\nestimated_time_hours range:", df['Estimated_Time_Hours'].min(), "-", df['Estimated_Time_Hours'].max())
print("attack_path_length range:", df['Attack_Path_Length'].min(), "-", df['Attack_Path_Length'].max())
print("kill_chain_count range:", df['kill_chain_count'].min(), "-", df['kill_chain_count'].max())

# any machines with no learning objectives?
no_obj = df['Learning_Objectives'].isna().sum() + (df['Learning_Objectives'].fillna('').str.strip() == '').sum()
print("\nmachines with no learning objectives:", no_obj)

# objective count per machine
obj_per_machine = df['Learning_Objectives'].fillna('').apply(lambda s: len([x for x in s.split(';') if x.strip()]))
print("objectives per machine - min/max/mean:", obj_per_machine.min(), "/", obj_per_machine.max(), "/", round(obj_per_machine.mean(), 2))

# the machine_features matrix - any all-zero rows?
all_zero_rows = (machine_features.sum(axis=1) == 0).sum()
print("\nall-zero machine rows:", all_zero_rows)

# any feature columns that are all zeros (useless features)?
all_zero_cols = (machine_features.sum(axis=0) == 0).sum()
print("all-zero feature columns:", all_zero_cols)

any duplicate machine names? 0

estimated_time_hours range: 0.5 - 4.5
attack_path_length range: 3 - 13
kill_chain_count range: 3 - 7

machines with no learning objectives: 0
objectives per machine - min/max/mean: 3 / 13 / 7.39

all-zero machine rows: 0
all-zero feature columns: 0


In [38]:
# building the student feature vector now, has to match machine columns exactly so the model can compare them

feature_columns = list(machine_features.columns)
print("total feature columns:", len(feature_columns))

# mapping the time bin to actual hours (midpoint of each range)
time_to_hours = {
    '<1 hour':    0.5,
    '1-2 hours':  1.5,
    '2-3 hours':  2.5,
    '3-4 hours':  3.5,
    '4+ hours':   4.5
}

# skill level is a new field for v2, will be used during training data generation
skill_to_num = {
    'Beginner':     1,
    'Intermediate': 2,
    'Advanced':     3
}

def build_student_vector(difficulty, attack_categories, os_pref,
                         learning_objectives=None, estimated_time=None,
                         skill_level=None):
    # turning the student form input into a row that matches the machine feature columns
    
    vec = pd.Series(0.0, index=feature_columns)
    
    # difficulty goes in directly
    vec['diff_num'] = diff_map.get(difficulty, 2)
    
    # estimated time defaults to 2.5h if not given
    vec['Estimated_Time_Hours'] = time_to_hours.get(estimated_time, 2.5)
    
    # student doesnt input attack path length or kill chain count so picking sensible defaults based on difficulty
    path_default = {'Easy': 4, 'Medium': 6, 'Hard': 9}
    chain_default = {'Easy': 3, 'Medium': 5, 'Hard': 7}
    vec['Attack_Path_Length'] = path_default.get(difficulty, 6)
    vec['kill_chain_count'] = chain_default.get(difficulty, 5)
    
    # attack category can be 1 or 2 values
    if isinstance(attack_categories, str):
        attack_categories = [attack_categories]
    for cat in attack_categories:
        col = 'cat_' + cat
        if col in vec.index:
            vec[col] = 1
    
    # os is single select
    os_col = 'os_' + os_pref
    if os_col in vec.index:
        vec[os_col] = 1
    
    # learning objectives are optional, 0 to 3 of them
    if learning_objectives:
        for obj in learning_objectives:
            col = 'obj_' + obj
            if col in vec.index:
                vec[col] = 1
    
    return vec


# testing with a sample student to make sure the function works
test_student = build_student_vector(
    difficulty='Easy',
    attack_categories=['Web Exploitation'],
    os_pref='Linux',
    learning_objectives=['SQL Injection', 'Credential Extraction'],
    estimated_time='1-2 hours',
    skill_level='Beginner'
)

print("\ntest student vector built ok, shape:", test_student.shape)
print("\nnon-zero values in this student:")
print(test_student[test_student != 0])

total feature columns: 111

test student vector built ok, shape: (111,)

non-zero values in this student:
diff_num                     1.0
Estimated_Time_Hours         1.5
Attack_Path_Length           4.0
kill_chain_count             3.0
cat_Web Exploitation         1.0
os_Linux                     1.0
obj_Credential Extraction    1.0
obj_SQL Injection            1.0
dtype: float64


In [42]:
# generating synthetic training data
# for each fake student we compute relevance against every machine by counting
# how many of their preferences the machine actually satisfies

import random

random.seed(42)
np.random.seed(42)

# pulling value options from the dataset so synthetic students are realistic
all_difficulties = ['Easy', 'Medium', 'Hard']
all_categories   = list(df['Attack_Category'].unique())
all_os           = list(df['OS'].unique())
all_times        = list(df['Estimated_Time'].dropna().unique())
all_skills       = ['Beginner', 'Intermediate', 'Advanced']
unique_obj_list  = sorted(set(all_objs))

print("possible values:")
print("  difficulties:", all_difficulties)
print("  categories:", len(all_categories))
print("  os options:", all_os)
print("  time bins:", len(all_times))
print("  objectives:", len(unique_obj_list))

# precomputing machine-side arrays so comparisons are fast
machine_difficulty = df['Difficulty'].values
machine_category   = df['Attack_Category'].values
machine_os         = df['OS'].values
machine_time       = df['Estimated_Time'].values
machine_objs_sets  = [set(o.strip() for o in str(s).split(';') if o.strip()) 
                      for s in df['Learning_Objectives']]

machine_features_array = machine_features.values
machine_feature_cols   = ['m_' + c for c in machine_features.columns]
n_machines = len(df)

# generating synthetic students
NUM_STUDENTS = 2000
synthetic_students = []

for sid in range(NUM_STUDENTS):
    diff = random.choice(all_difficulties)
    n_cats = 1 if random.random() < 0.7 else 2
    cats = random.sample(all_categories, min(n_cats, len(all_categories)))
    os_pick = random.choices(all_os, weights=[10, 1, 1])[0]
    n_objs = random.choices([0, 1, 2, 3], weights=[2, 3, 3, 2])[0]
    objs = random.sample(unique_obj_list, n_objs) if n_objs > 0 else []
    time_pick = random.choice(all_times) if random.random() < 0.6 else None
    skill = random.choice(all_skills) if random.random() < 0.5 else None
    
    synthetic_students.append({
        'student_id': sid,
        'difficulty': diff,
        'categories': cats,
        'os': os_pick,
        'objectives': objs,
        'time': time_pick,
        'skill': skill
    })

print("\ngenerated", len(synthetic_students), "synthetic students")
print("building training pairs...")

student_feature_cols = ['s_' + c for c in machine_features.columns]

student_blocks = []
machine_blocks = []
relevance_list = []
student_ids = []
machine_ids = df['Machine_ID'].values

for student in synthetic_students:
    svec = build_student_vector(
        difficulty=student['difficulty'],
        attack_categories=student['categories'],
        os_pref=student['os'],
        learning_objectives=student['objectives'],
        estimated_time=student['time'],
        skill_level=student['skill']
    ).values
    
    # repeat the student vector for every machine
    student_block = np.tile(svec, (n_machines, 1))
    student_blocks.append(student_block)
    machine_blocks.append(machine_features_array)
    
    student_cats = student['categories']
    student_objs = set(student['objectives'])
    has_time     = student['time'] is not None
    
    # difficulty match across all machines
    diff_match = (machine_difficulty == student['difficulty']).astype(int)
    
    # os match
    os_match = (machine_os == student['os']).astype(int)
    
    # category match (mixed counts as web or network)
    cat_matches = np.zeros(n_machines, dtype=int)
    cat_totals = 0
    for cat in student_cats:
        match = (machine_category == cat)
        if cat in ['Web Exploitation', 'Network Exploitation']:
            match = match | (machine_category == 'Mixed (Web + Network)')
        cat_matches += match.astype(int)
        cat_totals += 1
    
    # objectives match
    obj_matches = np.zeros(n_machines, dtype=int)
    for obj in student_objs:
        for i in range(n_machines):
            if obj in machine_objs_sets[i]:
                obj_matches[i] += 1
    
    # time match
    if has_time:
        time_match = (machine_time == student['time']).astype(int)
    else:
        time_match = np.zeros(n_machines, dtype=int)
    
    # total preferences given by this student
    n_total = 1 + cat_totals + 1 + len(student_objs) + (1 if has_time else 0)
    
    total_matched = diff_match + cat_matches + os_match + obj_matches + time_match
    relevance = total_matched / n_total
    
    relevance_list.append(relevance)
    student_ids.append(np.full(n_machines, student['student_id']))

print("assembling final dataframe...")
student_matrix  = np.vstack(student_blocks)
machine_matrix  = np.vstack(machine_blocks)
all_relevance   = np.concatenate(relevance_list)
all_student_ids = np.concatenate(student_ids)
all_machine_ids = np.tile(machine_ids, NUM_STUDENTS)

student_df = pd.DataFrame(student_matrix, columns=student_feature_cols)
machine_df = pd.DataFrame(machine_matrix, columns=machine_feature_cols)

training_data = pd.concat([student_df, machine_df], axis=1)
training_data['student_id'] = all_student_ids
training_data['machine_id'] = all_machine_ids
training_data['relevance']  = all_relevance

print("\ntraining data shape:", training_data.shape)
print("\nrelevance distribution:")
print(training_data['relevance'].describe())

# saving full parquet and a small csv sample
training_data.to_parquet(os.path.join(MODELS_DIR_V2, 'training_data.parquet'), index=False)
training_data.head(1000).to_csv(os.path.join(MODELS_DIR_V2, 'training_data_sample.csv'), index=False)

print("\nsaved training_data.parquet and training_data_sample.csv")

possible values:
  difficulties: ['Easy', 'Medium', 'Hard']
  categories: 5
  os options: ['Linux', 'FreeBSD', 'Windows']
  time bins: 5
  objectives: 99

generated 2000 synthetic students
building training pairs...
assembling final dataframe...

training data shape: (610000, 225)

relevance distribution:
count    610000.000000
mean          0.316585
std           0.176848
min           0.000000
25%           0.200000
50%           0.285714
75%           0.400000
max           1.000000
Name: relevance, dtype: float64


ArrowKeyError: A type extension with name pandas.period already defined

In [43]:
# saving as csv instead since parquet has a version issue
print("saving full training data as csv...")
training_data.to_csv(os.path.join(MODELS_DIR_V2, 'training_data.csv'), index=False)
training_data.head(1000).to_csv(os.path.join(MODELS_DIR_V2, 'training_data_sample.csv'), index=False)
print("saved training_data.csv (full) and training_data_sample.csv")
print("file size will be around 200-300mb but pandas reads it fine")

saving full training data as csv...
saved training_data.csv (full) and training_data_sample.csv
file size will be around 200-300mb but pandas reads it fine


In [44]:
# splitting the data into train and test
# train will be used to fit the models, test held out to measure real performance

from sklearn.model_selection import train_test_split

# pulling the relevance column out as the label, dropping id columns and the label itself from features
feature_cols = [c for c in training_data.columns if c not in ['student_id', 'machine_id', 'relevance']]
X = training_data[feature_cols].values
y = training_data['relevance'].values

print("feature matrix X shape:", X.shape)
print("label vector y shape:", y.shape)
print("number of features:", len(feature_cols))

# 80 percent train, 20 percent test
# random state set so the split is reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("\nafter splitting:")
print("  X_train:", X_train.shape, "y_train:", y_train.shape)
print("  X_test: ", X_test.shape, "y_test: ", y_test.shape)

# also keeping track of which student each row belongs to in test set
# we will need this later for ranking metrics (ndcg, precision@5) which work per-student
test_indices = training_data.iloc[X_train.shape[0]:].index  # not used directly but for context

# the proper way to know test student IDs is to do the split on the dataframe itself
train_df, test_df = train_test_split(training_data, test_size=0.20, random_state=42)
print("\ntrain_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("unique students in test set:", test_df['student_id'].nunique())

feature matrix X shape: (610000, 222)
label vector y shape: (610000,)
number of features: 222

after splitting:
  X_train: (488000, 222) y_train: (488000,)
  X_test:  (122000, 222) y_test:  (122000,)

train_df shape: (488000, 225)
test_df shape: (122000, 225)
unique students in test set: 2000


In [45]:
# generating balanced synthetic training data
# mixing random students with guided students so we get a mix of perfect/good/poor matches
# this ensures the model sees enough high-relevance examples to learn what makes a great match

import random

random.seed(42)
np.random.seed(42)

all_difficulties = ['Easy', 'Medium', 'Hard']
all_categories   = list(df['Attack_Category'].unique())
all_os           = list(df['OS'].unique())
all_times        = list(df['Estimated_Time'].dropna().unique())
all_skills       = ['Beginner', 'Intermediate', 'Advanced']
unique_obj_list  = sorted(set(all_objs))

# precomputing machine arrays for fast lookups
machine_difficulty = df['Difficulty'].values
machine_category   = df['Attack_Category'].values
machine_os         = df['OS'].values
machine_time       = df['Estimated_Time'].values
machine_objs_sets  = [set(o.strip() for o in str(s).split(';') if o.strip()) 
                      for s in df['Learning_Objectives']]

machine_features_array = machine_features.values
machine_feature_cols   = ['m_' + c for c in machine_features.columns]
n_machines = len(df)

# generating synthetic students
# half are random (general distribution) and half are guided (designed to match real machines)
NUM_RANDOM = 1000
NUM_GUIDED = 1000
NUM_STUDENTS = NUM_RANDOM + NUM_GUIDED

synthetic_students = []

# random students - same as before
for sid in range(NUM_RANDOM):
    diff = random.choice(all_difficulties)
    n_cats = 1 if random.random() < 0.7 else 2
    cats = random.sample(all_categories, min(n_cats, len(all_categories)))
    os_pick = random.choices(all_os, weights=[10, 1, 1])[0]
    n_objs = random.choices([0, 1, 2, 3], weights=[2, 3, 3, 2])[0]
    objs = random.sample(unique_obj_list, n_objs) if n_objs > 0 else []
    time_pick = random.choice(all_times) if random.random() < 0.6 else None
    skill = random.choice(all_skills) if random.random() < 0.5 else None
    synthetic_students.append({
        'student_id': sid, 'difficulty': diff, 'categories': cats, 'os': os_pick,
        'objectives': objs, 'time': time_pick, 'skill': skill
    })

# guided students - we pick a real machine, then build a student around it
# this guarantees at least some high-relevance examples per student
for i in range(NUM_GUIDED):
    sid = NUM_RANDOM + i
    
    # pick a random real machine to base this student on
    target_machine = df.sample(1).iloc[0]
    target_objs = [o.strip() for o in str(target_machine['Learning_Objectives']).split(';') if o.strip()]
    
    # build a student whose preferences mostly match this machine
    diff = target_machine['Difficulty']
    
    # category from the machine, sometimes add a second random category
    cats = [target_machine['Attack_Category']]
    if random.random() < 0.3 and len(all_categories) > 1:
        extra = random.choice([c for c in all_categories if c != target_machine['Attack_Category']])
        cats.append(extra)
    
    os_pick = target_machine['OS']
    
    # pick 1 to 3 objectives from this machine's objective list
    if target_objs:
        n_objs = random.choices([1, 2, 3], weights=[2, 3, 3])[0]
        n_objs = min(n_objs, len(target_objs))
        objs = random.sample(target_objs, n_objs)
    else:
        objs = []
    
    # time from the machine sometimes
    time_pick = target_machine['Estimated_Time'] if random.random() < 0.7 else None
    skill = random.choice(all_skills) if random.random() < 0.5 else None
    
    synthetic_students.append({
        'student_id': sid, 'difficulty': diff, 'categories': cats, 'os': os_pick,
        'objectives': objs, 'time': time_pick, 'skill': skill
    })

print(f"generated {NUM_RANDOM} random + {NUM_GUIDED} guided = {NUM_STUDENTS} students total")
print("building training pairs...")

student_feature_cols = ['s_' + c for c in machine_features.columns]

student_blocks = []
machine_blocks = []
relevance_list = []
student_ids = []
machine_ids = df['Machine_ID'].values

for student in synthetic_students:
    svec = build_student_vector(
        difficulty=student['difficulty'], attack_categories=student['categories'],
        os_pref=student['os'], learning_objectives=student['objectives'],
        estimated_time=student['time'], skill_level=student['skill']
    ).values
    student_blocks.append(np.tile(svec, (n_machines, 1)))
    machine_blocks.append(machine_features_array)
    
    student_cats = student['categories']
    student_objs = set(student['objectives'])
    has_time = student['time'] is not None
    
    diff_match = (machine_difficulty == student['difficulty']).astype(int)
    os_match = (machine_os == student['os']).astype(int)
    cat_matches = np.zeros(n_machines, dtype=int)
    cat_totals = 0
    for cat in student_cats:
        match = (machine_category == cat)
        if cat in ['Web Exploitation', 'Network Exploitation']:
            match = match | (machine_category == 'Mixed (Web + Network)')
        cat_matches += match.astype(int)
        cat_totals += 1
    obj_matches = np.zeros(n_machines, dtype=int)
    for obj in student_objs:
        for i in range(n_machines):
            if obj in machine_objs_sets[i]:
                obj_matches[i] += 1
    if has_time:
        time_match = (machine_time == student['time']).astype(int)
    else:
        time_match = np.zeros(n_machines, dtype=int)
    n_total = 1 + cat_totals + 1 + len(student_objs) + (1 if has_time else 0)
    total_matched = diff_match + cat_matches + os_match + obj_matches + time_match
    relevance = total_matched / n_total
    relevance_list.append(relevance)
    student_ids.append(np.full(n_machines, student['student_id']))

print("assembling final dataframe...")
student_matrix  = np.vstack(student_blocks)
machine_matrix  = np.vstack(machine_blocks)
all_relevance   = np.concatenate(relevance_list)
all_student_ids = np.concatenate(student_ids)
all_machine_ids = np.tile(machine_ids, NUM_STUDENTS)

student_df = pd.DataFrame(student_matrix, columns=student_feature_cols)
machine_df = pd.DataFrame(machine_matrix, columns=machine_feature_cols)

training_data = pd.concat([student_df, machine_df], axis=1)
training_data['student_id'] = all_student_ids
training_data['machine_id'] = all_machine_ids
training_data['relevance']  = all_relevance

print("\ntraining data shape:", training_data.shape)
print("\nrelevance distribution:")
print(training_data['relevance'].describe().round(3))

# checking the balance by bucket
print("\nbalance by bucket:")
buckets = pd.cut(training_data['relevance'], 
                 bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.01],
                 labels=['poor', 'low', 'med', 'good', 'excellent'])
print(buckets.value_counts().sort_index())

# saving
print("\nsaving...")
training_data.to_csv(os.path.join(MODELS_DIR_V2, 'training_data.csv'), index=False)
training_data.head(1000).to_csv(os.path.join(MODELS_DIR_V2, 'training_data_sample.csv'), index=False)
print("saved training_data.csv and training_data_sample.csv")

generated 1000 random + 1000 guided = 2000 students total
building training pairs...
assembling final dataframe...

training data shape: (610000, 225)

relevance distribution:
count    610000.000
mean          0.404
std           0.203
min           0.000
25%           0.250
50%           0.400
75%           0.500
max           1.000
Name: relevance, dtype: float64

balance by bucket:
relevance
poor         132263
low          210018
med          179845
good          70365
excellent     17509
Name: count, dtype: int64

saving...
saved training_data.csv and training_data_sample.csv
